# P1: Matnga Ishlov Berish Konveyeri

Bugun bitta kichik NLP pipeline quramiz: xom matndan tozalangan tokenlar, keyin BoW va TF-IDF matritsalari.

**80 daqiqalik oqim:**
1. Yakuniy manzilni ko'ramiz.
2. Matnni tokenlarga ajratamiz.
3. Stop-so'z va qo'shimchalarni kamaytiramiz.
4. Natijani vizual tekshiramiz.
5. BoW / TF-IDF vektorlarini quramiz.
6. Oxirida shu pipeline ni kichik `TextPreprocessor` klassiga o'raymiz.

Hugging Face kerak emas. Kaggle notebook + oddiy Python kutubxonalari yetarli.


## Ishlatiladigan kutubxonalar

Bu amaliyotda og'ir frameworklar kerak emas. Har bir paketning vazifasi kichik:

- `pathlib` — Kaggle yoki local fayl yo'llarini topish;
- `re` — o'zbekcha apostrofli so'zlar uchun regex tokenizer;
- `Counter` — token chastotasi va corpus stop-so'zlarini hisoblash;
- `math` / `numpy` — TF-IDF hisoblari va matritsa tekshiruvlari;
- `pandas` — oraliq natijalarni jadval qilib ko'rsatish;
- `matplotlib` — lug'at hajmi, sparse matrix va TF-IDF top-so'zlarini chizish;
- `scikit-learn` — `CountVectorizer` va `TfidfVectorizer`.

Hugging Face bu darsda ishlatilmaydi; u keyingi dataset/model darslari uchun kerak bo'ladi.


In [ ]:
# Run only: setup
from pathlib import Path
from collections import Counter
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 90)

OFFLINE_FALLBACK = True
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [ ]:
# Run only: Kaggle/local data loader
FALLBACK_TEXTS = [
    "O'zbekistonda sun'iy intellekt va NLP texnologiyalari tez rivojlanmoqda.",
    "Toshkentdagi universitetlarda dasturlash va ma'lumotlar tahlili o'qitilmoqda.",
    "Yoshlar yangi texnologiyalarni o'rganib, foydali loyihalar yaratmoqda.",
    "Matnga ishlov berish tokenizatsiya, stop-so'zlarni olib tashlash va stemmingdan boshlanadi.",
    "Axborot qidiruv tizimlari hujjatlarni tez topish uchun TF-IDF kabi usullardan foydalanadi.",
]


def find_data_file(filename="uz_news_mini.txt"):
    candidates = []
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates.extend(kaggle_root.rglob(filename))
    candidates.extend([
        Path("practices/d02_checkpoints") / filename,
        Path("../practices/d02_checkpoints") / filename,
        Path("d02_checkpoints") / filename,
    ])
    for path in candidates:
        if path.exists():
            return path
    return None

DATA_PATH = find_data_file()
if DATA_PATH is not None:
    raw_texts = [line.strip() for line in DATA_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
    source_label = str(DATA_PATH)
elif OFFLINE_FALLBACK:
    raw_texts = FALLBACK_TEXTS
    source_label = "embedded fallback texts"
else:
    raise FileNotFoundError("uz_news_mini.txt topilmadi. Kaggle Dataset biriktirilganini tekshiring.")

print(f"Manba: {source_label}")
print(f"Hujjatlar soni: {len(raw_texts)}")
print("Namuna:", raw_texts[0])


## Yakuniy manzil

Bugungi pipeline shunday ishlaydi:

`raw text -> normalize apostrophes -> tokenize -> remove stopwords -> stem -> BoW / TF-IDF`

Har bosqichdan keyin kichik jadval yoki grafik bilan tekshiramiz.


In [ ]:
# Run only: small display helpers

def show_table(rows, columns=None):
    return pd.DataFrame(rows, columns=columns)


def plot_vocab_change(raw_tokens, clean_tokens):
    values = [len(set(raw_tokens)), len(set(clean_tokens))]
    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    bars = ax.bar(["xom lug'at", "tozalangan lug'at"], values, color=["#4C78A8", "#F58518"])
    ax.set_ylabel("unikal tokenlar")
    ax.set_title("Preprocessing lug'at hajmini kamaytiradi")
    ax.bar_label(bars, padding=3)
    plt.show()


def plot_sparse_preview(matrix, title, max_rows=10, max_cols=20):
    arr = matrix[:max_rows, :max_cols].toarray()
    fig, ax = plt.subplots(figsize=(8, 3.6))
    im = ax.imshow(arr, aspect="auto", cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("birinchi featurelar")
    ax.set_ylabel("hujjatlar")
    fig.colorbar(im, ax=ax, fraction=0.03)
    plt.show()


def plot_top_tfidf_terms(vec, X, doc_id=0, top_n=8):
    terms = np.array(vec.get_feature_names_out())
    row = X[doc_id].toarray().ravel()
    top_idx = row.argsort()[::-1][:top_n]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.barh(terms[top_idx][::-1], row[top_idx][::-1], color="#54A24B")
    ax.set_title(f"{doc_id + 1}-hujjat: eng muhim TF-IDF so'zlar")
    ax.set_xlabel("TF-IDF vazni")
    plt.show()


## 1. Apostrof va tokenlar

O'zbek lotin yozuvida apostrof muhim: `o'zbek`, `g'oya`, `bo'yicha`. Avval turli apostroflarni bitta shaklga keltiramiz, keyin regex bilan token olamiz.


In [ ]:
_APOSTROPHE_RE = re.compile(r"['’‘]")
_TOKEN_RE = re.compile(r"[a-z][a-z']*")


def normalize_apostrophes(text: str) -> str:
    return _APOSTROPHE_RE.sub("'", text).lower()


def uz_tokenize(text: str) -> list[str]:
    # TODO 1: normalize_apostrophes(text) ni chaqiring
    # TODO 2: _TOKEN_RE.findall(...) bilan tokenlar ro'yxatini qaytaring
    pass


In [ ]:
samples = [
    "O'zbekistonda NLP rivojlanmoqda!",
    "O’zbek tili bo’yicha yangi loyiha.",
    raw_texts[0],
]

token_rows = []
for text in samples:
    toks = uz_tokenize(text)
    token_rows.append({"xom matn": text, "tokenlar": toks})

assert all(isinstance(row["tokenlar"], list) and row["tokenlar"] for row in token_rows), (
    "uz_tokenize() bo'sh bo'lmagan list qaytarishi kerak."
)
show_table(token_rows)


## 2. Stop-so'zlarni olib tashlash

Stop-so'zlar tez-tez uchraydi, lekin ko'pincha mavzuni ajratishga kam yordam beradi: `va`, `bu`, `bilan`, `da`, `ni`, `ga`.


In [ ]:
UZ_STOPWORDS: set[str] = {
    "va", "bu", "bir", "bilan", "da", "ni", "ga", "dan",
    "ham", "uchun", "bo'lgan", "bo'lib", "bo'ldi", "o'z",
    "ular", "u", "men", "biz", "siz", "edi", "ekan", "deb",
    "lekin", "ammo", "yoki", "agar", "chunki", "hali",
    "ko'p", "oz", "shunday", "shu", "esa", "endi",
    "bor", "yo'q", "kerak", "mumkin", "bo'lsa", "bo'lishi",
}


def filter_stopwords(tokens: list[str], stopwords: set[str]) -> list[str]:
    # TODO: stopword bo'lmagan va uzunligi kamida 2 bo'lgan tokenlarni qaytaring
    pass


In [ ]:
raw_tokens = uz_tokenize(raw_texts[0])
filtered_tokens = filter_stopwords(raw_tokens, UZ_STOPWORDS)

assert filtered_tokens is not None, "filter_stopwords() None qaytardi."
assert len(filtered_tokens) <= len(raw_tokens), "Filtrlangan tokenlar xom tokenlardan ko'p bo'lmasligi kerak."
assert all(t not in UZ_STOPWORDS for t in filtered_tokens), "Stop-so'z qoldi."
assert all(len(t) >= 2 for t in filtered_tokens), "Juda qisqa token qoldi."

show_table([
    {"bosqich": "tokenize", "soni": len(raw_tokens), "natija": raw_tokens},
    {"bosqich": "stopword filter", "soni": len(filtered_tokens), "natija": filtered_tokens},
])


## 3. Sodda stemming

O'zbek uchun tayyor universal stemmer yo'q. Bu darsda pedagogik, sodda suffix stripping ishlatamiz. Maqsad mukammal lingvistika emas, balki featurelar sonini kamaytirish.


In [ ]:
_UZ_SUFFIXES: tuple[str, ...] = (
    "larning", "lardan", "larni", "larga", "larim", "laring", "lari",
    "imizdan", "ingizdan",
    "lilik", "ligi", "lik",
    "roqqa", "roq",
    "ishda", "ishi", "ish",
    "ning", "chi",
    "lar", "ni", "da", "dan", "ga",
)


def uz_stem(token: str) -> str:
    for suffix in sorted(_UZ_SUFFIXES, key=len, reverse=True):
        if token.endswith(suffix) and len(token) - len(suffix) >= 3:
            return token[:-len(suffix)]
    return token

stem_demo = ["yaxshi", "yaxshiroq", "yaxshilik", "texnologiyalari", "o'qituvchilar", "kitoblarni"]
show_table([{"token": w, "stem": uz_stem(w)} for w in stem_demo])


## 4. To'liq preprocessing pipeline

Endi uchta kichik qismni ulaymiz: `tokenize -> stopword filter -> stem`.


In [ ]:
def preprocess_doc(text: str) -> list[str]:
    # TODO 1: matnni tokenlarga ajrating
    # TODO 2: stop-so'zlarni filtrlang
    # TODO 3: har token uchun uz_stem() qo'llang va ro'yxatni qaytaring
    pass

# TODO: raw_texts dagi har bir hujjatga preprocess_doc() qo'llang
processed_corpus = None


In [ ]:
assert processed_corpus is not None, "processed_corpus None. List comprehension ishlating."
assert len(processed_corpus) == len(raw_texts), "Har bir hujjat qayta ishlanishi kerak."
assert all(isinstance(doc, list) and doc for doc in processed_corpus), "Har bir natija bo'sh bo'lmagan list bo'lishi kerak."

raw_all_tokens = [t.lower() for text in raw_texts for t in text.split()]
clean_all_tokens = [t for doc in processed_corpus for t in doc]

print("Birinchi hujjat:")
print("xom:", raw_texts[0])
print("tozalangan:", processed_corpus[0])
plot_vocab_change(raw_all_tokens, clean_all_tokens)


## 5. TF-IDF ni qo'lda tekshirish

Ma'ruzadagi kichik misolni kodda hisoblaymiz. Bu yerda formula soddalashtirilgan: `IDF(t)=ln(N/df(t))`.


In [ ]:
docs_l1 = ["nlp qiziq", "python foydali", "nlp foydali"]
N = len(docs_l1)


def compute_df(corpus: list[str]) -> dict[str, int]:
    df = {}
    for doc in corpus:
        for word in set(doc.split()):
            df[word] = df.get(word, 0) + 1
    return df


def tf(word: str, doc: str) -> int:
    return doc.split().count(word)


def idf(word: str, df: dict[str, int], n_docs: int) -> float:
    return math.log(n_docs / df[word])


df_l1 = compute_df(docs_l1)
rows = []
for word in ["nlp", "qiziq"]:
    value = tf(word, docs_l1[0]) * idf(word, df_l1, N)
    rows.append({"word": word, "df": df_l1[word], "idf": round(idf(word, df_l1, N), 3), "tf-idf D1": round(value, 3)})

assert abs(rows[0]["tf-idf D1"] - 0.405) < 1e-3
assert abs(rows[1]["tf-idf D1"] - 1.099) < 1e-3
show_table(rows)


## 6. BoW va TF-IDF matritsalari

`scikit-learn` bizga matnlarni sonli feature matritsaga aylantirib beradi. Bu matritsalar keyingi darsdagi klassifikatorlarga kiradi.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

corpus_strings = [" ".join(doc) for doc in processed_corpus]

# TODO 1: CountVectorizer() yarating
# TODO 2: fit_transform(corpus_strings) chaqiring
vec_bow = None
X_bow = None


In [ ]:
assert vec_bow is not None and X_bow is not None, "CountVectorizer va X_bow ni yarating."
assert X_bow.shape[0] == len(raw_texts), "Qatorlar soni hujjatlar soniga teng bo'lishi kerak."
assert X_bow.shape[1] > 0, "Lug'at bo'sh chiqdi."

print(f"BoW matritsasi: {X_bow.shape[0]} hujjat x {X_bow.shape[1]} so'z")
print("Birinchi 10 feature:", list(vec_bow.get_feature_names_out())[:10])
plot_sparse_preview(X_bow, "BoW sparse matritsa preview")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO 1: TfidfVectorizer() yarating
# TODO 2: fit_transform(corpus_strings) chaqiring
vec_tfidf = None
X_tfidf = None


In [ ]:
assert vec_tfidf is not None and X_tfidf is not None, "TfidfVectorizer va X_tfidf ni yarating."
assert X_tfidf.shape == X_bow.shape, "TF-IDF va BoW bir xil shaklda bo'lishi kerak."

row_norms = np.asarray(X_tfidf.power(2).sum(axis=1)).ravel()
assert np.allclose(row_norms, 1.0, atol=1e-6), "Default TF-IDF qatorlari L2 normaga kelishi kerak."

print(f"TF-IDF matritsasi: {X_tfidf.shape[0]} hujjat x {X_tfidf.shape[1]} so'z")
plot_top_tfidf_terms(vec_tfidf, X_tfidf, doc_id=0, top_n=8)


## 7. Mini mustaqil mashq (agar vaqt qolsa)

Quyidagi uchta matnga shu pipeline ni qo'llang. Bu qism bonus: 80 daqiqalik darsda vaqt yetmasa, uyga vazifa bo'lishi mumkin.


In [ ]:
my_texts = [
    "O'zbekistonda sun'iy intellekt rivojlanmoqda.",
    "Yoshlar dasturlash va texnologiyalarni o'rganmoqda.",
    "Milliy texnologiyalar markazi yangi loyihalar yaratmoqda.",
]

# BONUS TODO: preprocess_doc + TfidfVectorizer ni qo'llang
my_processed = None
my_vec_tfidf = None
my_X_tfidf = None

if my_processed is None:
    print("Bonus mashq hozircha o'tkazib yuborildi.")
else:
    assert my_X_tfidf.shape[0] == len(my_texts)
    plot_top_tfidf_terms(my_vec_tfidf, my_X_tfidf, doc_id=0, top_n=5)


## 8. Yengil capstone bog'lanish

Bugungi pipeline ni keyingi darslarda qayta ishlatish uchun kichik klassga o'raymiz. Hozircha papka arxitekturasini chuqur ko'rmaymiz; faqat g'oya: **bitta qayta ishlatiladigan preprocessor**.


In [ ]:
class TextPreprocessor:
    def __init__(self) -> None:
        self.stopwords = set(UZ_STOPWORDS)

    def preprocess(self, text: str) -> list[str]:
        if not isinstance(text, str) or not text.strip():
            raise ValueError("text bo'sh bo'lmasligi kerak")
        tokens = uz_tokenize(text)
        tokens = filter_stopwords(tokens, self.stopwords)
        return [uz_stem(t) for t in tokens]

    def preprocess_batch(self, texts: list[str]) -> list[list[str]]:
        return [self.preprocess(text) for text in texts]

    def fit_stopwords(self, texts: list[str], max_df: float = 0.85) -> None:
        threshold = max(1, int(len(texts) * max_df))
        df = Counter()
        for text in texts:
            df.update(set(uz_tokenize(text)))
        self.stopwords.update(word for word, count in df.items() if count >= threshold)


In [ ]:
tp = TextPreprocessor()
example = "O'zbek tilida tabiiy tilni qayta ishlash texnologiyalari"
print(tp.preprocess(example))

assert isinstance(tp.preprocess(example), list)
assert len(tp.preprocess_batch([example, raw_texts[0]])) == 2
try:
    tp.preprocess("")
    raise AssertionError("Bo'sh string ValueError berishi kerak edi.")
except ValueError:
    pass

print("TextPreprocessor tayyor: bugungi pipeline klassga o'raldi.")


## Yakun

Bugun siz quyidagilarni qurdingiz:

- o'zbek matni uchun apostrofga sezgir tokenizer;
- stop-so'z filtri;
- sodda suffix-stemmer;
- preprocessing pipeline;
- BoW va TF-IDF matritsalari;
- keyingi darslarda ishlatiladigan `TextPreprocessor` klassi.

Keyingi darsda shu vektorlar ustida sentiment klassifikator quramiz.
